# 0. 深度学习训练核心基础 (Deep Learning Training Fundamentals)

> 🕐 预估学习时间：50分钟

本节是所有大模型（LLM）训练技术的前置基础。无论是 LoRA 微调、混合精度训练，还是上下文扩展、分布式训练，都建立在本节介绍的核心机制之上。

掌握这些基础后，你将能够：
- 理解自动微分如何驱动模型学习
- 选择合适的优化器与学习率策略
- 应用正则化与归一化技术稳定训练

## 本节内容
1. 反向传播与自动微分
2. 优化器对比（SGD / Adam / AdamW）
3. 学习率调度（Warmup + Cosine Decay + WSD）
4. 正则化与归一化（Dropout / Label Smoothing / LayerNorm vs RMSNorm）
5. 课后思考题

## 1. 反向传播与自动微分

**自动微分（Autograd）** 是 PyTorch 的核心机制，通过动态计算图（Define-by-Run）自动计算梯度。

**核心概念**：
- **计算图**：有向无环图（DAG），节点是张量操作，边是数据依赖
- **链式法则**：∂L/∂x = ∂L/∂y · ∂y/∂x，反向传播逐层应用
- **grad_fn**：每个张量记录产生它的操作，构成反向传播路径
- **requires_grad**：标记需要计算梯度的张量

**反向模式自动微分**：从损失（标量）出发，反向遍历计算图，对每个节点计算 VJP（向量-雅可比积），适合神经网络这种输出维度远小于输入维度的场景。

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

print('=== Backpropagation & Autograd ===')

# Simple model: 2-layer MLP
model = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 1))
X = torch.randn(4, 8)
y = torch.randn(4, 1)

# Forward pass builds the computation graph dynamically
y_pred = model(X)
loss = ((y_pred - y) ** 2).mean()

print(f'Loss: {loss.item():.4f}')
print(f'Loss grad_fn: {loss.grad_fn}')
print(f'y_pred grad_fn: {y_pred.grad_fn}')

# torch.autograd.grad: explicit gradient computation (does not store in .grad)
w0 = model[0].weight
g = torch.autograd.grad(loss, w0, retain_graph=True)[0]
print(f'\ntorch.autograd.grad for w0: grad_norm={g.norm():.4f}')

# backward(): accumulates gradients in .grad
loss.backward()
print(f'backward() for w0: grad_norm={model[0].weight.grad.norm():.4f}')
print(f'Match: {torch.allclose(g, model[0].weight.grad)}')

print(f'\nGradient inspection (all parameters):')
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f'  {name:20s}: shape={str(list(param.shape)):12s} grad_norm={param.grad.norm():.4f}')

# requires_grad demonstration
x = torch.randn(3, requires_grad=True)
y2 = (x ** 2).sum()
y2.backward()
print(f'\nSimple example: x={x.data.tolist()}, dy/dx={x.grad.tolist()}')
print(f'  (gradient of x^2 is 2x)')

print(f'\nKey: PyTorch builds the computation graph dynamically during forward pass.')
print(f'backward() traverses the graph in reverse, applying chain rule at each node.')
print(f'torch.autograd.grad computes gradients explicitly without storing in .grad.')

## 2. 优化器对比 (SGD vs Adam vs AdamW)

**SGD（随机梯度下降）**：θ ← θ - η·g，简单但收敛慢，需要精心调学习率

**SGD + Momentum**：v ← β·v + g，θ ← θ - η·v，动量加速收敛、减少震荡

**Adam（自适应矩估计）**：
- 维护一阶矩 m（梯度的指数移动平均）和二阶矩 v（梯度平方的指数移动平均）
- 自适应学习率：每个参数有独立的有效学习率 η / (√v + ε)
- 适合大多数深度学习任务

**AdamW（解耦权重衰减）**：
- **Adam 的权重衰减**：通过 L2 正则化实现，衰减项进入梯度，与自适应学习率耦合
- **AdamW 的权重衰减**：直接衰减参数 θ ← (1-ηλ)θ，与梯度更新解耦
- 解耦后正则化效果更稳定，是 LLM 训练的标准优化器（GPT、LLaMA、DeepSeek 等）

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)

print('=== Optimizer Comparison: SGD vs Adam vs AdamW ===')

# Regression task: y = X @ w + noise
X = torch.randn(128, 4)
true_w = torch.tensor([[2.0], [3.0], [-1.0], [0.5]])
y = X @ true_w + 0.1 * torch.randn(128, 1)

def make_model():
    return nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 1))

def train(opt_name, steps=100):
    torch.manual_seed(42)
    model = make_model()
    if opt_name == 'SGD':
        opt = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
    elif opt_name == 'Adam':
        opt = optim.Adam(model.parameters(), lr=1e-2, weight_decay=0.01)
    else:
        opt = optim.AdamW(model.parameters(), lr=1e-2, weight_decay=0.01)

    loss_fn = nn.MSELoss()
    final_loss = 0.0
    for step in range(steps):
        pred = model(X)
        loss = loss_fn(pred, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if (step + 1) % 25 == 0:
            print(f'  [{opt_name:6s}] step {step+1:3d}: loss={loss.item():.4f}')
        final_loss = loss.item()

    param_norm = sum(p.norm().item() ** 2 for p in model.parameters()) ** 0.5
    return final_loss, param_norm

print(f'\nTraining 100 steps on regression task (weight_decay=0.01 for Adam/AdamW):')
sgd_loss, sgd_norm = train('SGD')
adam_loss, adam_norm = train('Adam')
adamw_loss, adamw_norm = train('AdamW')

print(f'\nOptimizer   Final Loss   Param Norm')
print(f'SGD         {sgd_loss:>12.4f} {sgd_norm:>12.4f}')
print(f'Adam        {adam_loss:>12.4f} {adam_norm:>12.4f}')
print(f'AdamW       {adamw_loss:>12.4f} {adamw_norm:>12.4f}')

print(f'\nKey: AdamW decouples weight decay from gradient update, providing better')
print(f'regularization than Adam (smaller param norm at similar loss).')
print(f'AdamW is the standard optimizer for LLM training.')

## 3. 学习率调度

学习率调度在训练过程中动态调整学习率，平衡收敛速度与稳定性。

**常用策略**：
- **Warmup（预热）**：训练初期线性增大学习率，避免初期不稳定
  - 原因：Adam 的二阶矩 v 初始为 0，自适应学习率过大
  - 典型步数：总步数的 1-5%
- **Cosine Decay（余弦衰减）**：从峰值余弦衰减到接近 0，平滑下降
- **WSD（Warmup-Stable-Decay）**：预热 → 稳定 → 衰减，DeepSeek 系列使用
  - 稳定阶段保持峰值学习率，最后阶段快速衰减

**LLM 训练最常用**：Warmup + Cosine Decay

In [ ]:
import torch
import math

torch.manual_seed(42)

print('=== Learning Rate Schedules ===')

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, peak_lr, min_lr=0.0):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.peak_lr = peak_lr
        self.min_lr = min_lr
        self.step_count = 0

    def get_lr(self, step):
        if step < self.warmup_steps:
            return self.peak_lr * step / self.warmup_steps
        progress = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        return self.min_lr + 0.5 * (self.peak_lr - self.min_lr) * (1 + math.cos(math.pi * progress))

    def step(self):
        lr = self.get_lr(self.step_count)
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr
        self.step_count += 1
        return lr

class WSDScheduler:
    def __init__(self, optimizer, warmup_steps, stable_steps, total_steps, peak_lr, min_lr=0.0):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.stable_steps = stable_steps
        self.total_steps = total_steps
        self.peak_lr = peak_lr
        self.min_lr = min_lr
        self.step_count = 0

    def get_lr(self, step):
        if step < self.warmup_steps:
            return self.peak_lr * step / self.warmup_steps
        elif step < self.warmup_steps + self.stable_steps:
            return self.peak_lr
        else:
            decay_steps = self.total_steps - self.warmup_steps - self.stable_steps
            progress = (step - self.warmup_steps - self.stable_steps) / decay_steps
            return self.min_lr + 0.5 * (self.peak_lr - self.min_lr) * (1 + math.cos(math.pi * progress))

    def step(self):
        lr = self.get_lr(self.step_count)
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr
        self.step_count += 1
        return lr

# Dummy optimizer for demonstration
dummy = torch.nn.Linear(2, 2)
opt = torch.optim.SGD(dummy.parameters(), lr=1e-3)

total_steps = 1000
warmup_steps = 100
cos_sched = WarmupCosineScheduler(opt, warmup_steps, total_steps, peak_lr=1e-3)
wsd_sched = WSDScheduler(opt, warmup_steps, stable_steps=500, total_steps=total_steps, peak_lr=1e-3)

cos_lrs = [cos_sched.get_lr(s) for s in range(total_steps)]
wsd_lrs = [wsd_sched.get_lr(s) for s in range(total_steps)]

print(f'Total steps: {total_steps}, warmup: {warmup_steps}, peak_lr: 1e-3')
print(f'\n  Step    Cosine       WSD')
for s in list(range(0, total_steps, 100)) + [total_steps - 1]:
    print(f'{s:>6} {cos_lrs[s]:>10.6f} {wsd_lrs[s]:>10.6f}')

print(f'\nKey: Cosine decay smoothly reduces lr throughout training (most common in LLMs).')
print(f'WSD keeps lr at peak during stable phase, then decays sharply at the end.')
print(f'Warmup prevents early-training instability from small second moments in Adam.')

## 4. 正则化与归一化

**正则化**：防止过拟合，提升泛化能力
- **Dropout**：训练时随机置零神经元（概率 p），推理时不丢弃。等效于集成多个子网络
- **Weight Decay（权重衰减）**：限制参数范数，AdamW 中解耦实现
- **Label Smoothing（标签平滑）**：将硬标签 [0, 1] 软化为 [(1-α)/K, 1-α+α/K]，防止模型过度自信

**归一化**：稳定训练、加速收敛
- **LayerNorm**：y = (x - μ) / √(σ² + ε) × γ + β，对每个 token 的隐层维度归一化
- **RMSNorm**：y = x / √(mean(x²) + ε) × γ，省略均值减法，计算更快
- **现代 LLM 首选 RMSNorm**：LLaMA、Mistral、DeepSeek 等均使用

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

print('=== Regularization & Normalization ===')

# 1. Dropout effect
print('\n--- Dropout ---')
dropout = nn.Dropout(p=0.5)
x = torch.ones(4, 8)

dropout.train()
out_train = dropout(x)
print(f'Training mode: {out_train[0].tolist()}')
print(f'  Non-zero count: {(out_train != 0).sum().item()}/{out_train.numel()}')
print(f'  Values scaled by 1/(1-p)=2.0 to preserve expectation')

dropout.eval()
out_eval = dropout(x)
print(f'Eval mode: {out_eval[0].tolist()}')
print(f'  No dropping in eval mode')

# 2. Label smoothing
print('\n--- Label Smoothing ---')
target = torch.tensor([1, 0, 2])
hard_labels = F.one_hot(target, num_classes=3).float()
alpha = 0.1
smoothed = hard_labels * (1 - alpha) + alpha / 3
print(f'Hard labels:\n{hard_labels}')
print(f'Smoothed labels (alpha=0.1):\n{smoothed}')

# 3. LayerNorm vs RMSNorm
print('\n--- LayerNorm vs RMSNorm ---')

class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight

x = torch.randn(4, 8) * 3 + 2  # non-zero mean, large std
layer_norm = nn.LayerNorm(8)
rms_norm = RMSNorm(8)

ln_out = layer_norm(x)
rn_out = rms_norm(x)

print(f'Input:        mean={x.mean(dim=-1)[0].item():.4f}, std={x.std(dim=-1)[0].item():.4f}')
print(f'LayerNorm:    mean={ln_out.mean(dim=-1)[0].item():.4f}, std={ln_out.std(dim=-1)[0].item():.4f}')
print(f'RMSNorm:      mean={rn_out.mean(dim=-1)[0].item():.4f}, std={rn_out.std(dim=-1)[0].item():.4f}')

ln_params = sum(p.numel() for p in layer_norm.parameters())
rn_params = sum(p.numel() for p in rms_norm.parameters())
print(f'\nLayerNorm params: {ln_params} (weight + bias)')
print(f'RMSNorm params:   {rn_params} (weight only, no bias)')

print(f'\nKey: Dropout and label smoothing prevent overfitting and overconfidence.')
print(f'RMSNorm is faster than LayerNorm (no mean subtraction) and is the standard')
print(f'in modern LLMs (LLaMA, Mistral, DeepSeek).')

## 📝 课后思考题

1. **AdamW vs SGD**：在什么场景下应该选择 AdamW 而非 SGD？为什么 LLM 训练几乎都用 AdamW？（提示：考虑自适应学习率、解耦权重衰减、大batch训练）

2. **Warmup 的作用**：为什么训练初期需要 warmup？如果不使用 warmup 直接用峰值学习率，会发生什么？（提示：考虑 Adam 二阶矩的初始化）

3. **RMSNorm vs LayerNorm**：为什么现代 LLM（LLaMA、DeepSeek）选择 RMSNorm 而非 LayerNorm？在什么情况下 LayerNorm 可能更优？

4. **Label Smoothing 的权衡**：标签平滑能防止模型过度自信，但过大的平滑系数会损害性能。如何选择合适的平滑系数 α？（提示：考虑任务难度和数据质量）